### Vacabulary building and True Ingredient Matching

In [8]:
# Optional if we wish to perform checks on ingredient detection
import json
import pandas as pd
from pathlib import Path

# Load training + validation subsets
with open("data/mini_data/train_recipes_stratified.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("data/mini_data_val/val_recipes_stratified.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

# Combine and label the source
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
train_df["split"] = "train"
val_df["split"] = "val"
recipe_df = pd.concat([train_df, val_df], ignore_index=True)

# Strip .jpg for merging
recipe_df["image_id"] = recipe_df["image_id"].str.replace(".jpg", "", regex=False)

# Load layer2.json
with open("data/recipe1m_images/layer2.json", "r", encoding="utf-8") as f:
    layer2 = json.load(f)

# Map image_id to recipe ID using layer2
image_to_recipe = {}
for entry in layer2:
    recipe_id = entry["id"]
    for img in entry.get("images", []):
        img_id = img.get("id", "").replace(".jpg", "")
        image_to_recipe[img_id] = recipe_id

# Add recipe ID column to combined subset based on image ID
recipe_df["recipe_id"] = recipe_df["image_id"].map(image_to_recipe)

# Load det_ingrs.json
with open("data/recipe1m_images/det_ingrs.json", "r", encoding="utf-8") as f:
    det_data = json.load(f)
det_df = pd.DataFrame(det_data)

# Extract detected ingredients if needed
if "detected_ingredients" not in det_df.columns:
    det_df["detected_ingredients"] = det_df.apply(
        lambda row: [i["text"] for i, v in zip(row["ingredients"], row["valid"]) if v], axis=1
    )

# Merge on recipe ID
merged_df = pd.merge(
    recipe_df,
    det_df[["id", "detected_ingredients"]],
    how="inner",
    left_on="recipe_id",
    right_on="id"
)

print(f"Merged {len(merged_df)} entries (train + val)")
print(merged_df[["split", "image_id", "ingredients", "detected_ingredients"]].head())
merged_df.to_json("data/mini_data/recipes_with_detected.json", orient="records", indent=2)


Merged 11000 entries (train + val)
   split    image_id                                        ingredients  \
0  train  9acd86306d  [1 tablespoon chili powder or 1 tablespoon dri...   
1  train  95d414fd89  [1 14 cups water, 2 tablespoons margarine, 5 t...   
2  train  5b778f4af4  [1 pound ground chuck, 1 pound lean ground bee...   
3  train  01262d1091  [1 teaspoon ground cumin, 1 teaspoon ground ca...   
4  train  058a11a08c  [1 lb tomatoes, 12 onion, 6 garlic cloves, 14 ...   

                                detected_ingredients  
0  [chili powder, salt, sugar, garlic powder, oni...  
1      [water, margarine, sugar, salt, flour, flour]  
2  [ground chuck, lean ground beef, tomato sauce,...  
3  [ground cumin, ground cayenne pepper, ground t...  
4  [tomatoes, onions, garlic cloves, fresh basil,...  


In [9]:
import json
from collections import Counter
from pathlib import Path
import pickle
import pandas as pd
import os

# Load recipes
with open("data/mini_data/train_recipes_stratified.json", "r", encoding="utf-8") as f:
    recipes = json.load(f)

# Build vocab from all ingredient words
counter = Counter()
for r in recipes:
    for ing in r["ingredients"]:
        tokens = ing.lower().split()
        counter.update(tokens)

# Keep words with at least 2 occurrences
min_freq = 2
vocab = ["<pad>", "<unk>"]
vocab += [word for word, freq in counter.items() if freq >= min_freq]

word2idx = {w: i for i, w in enumerate(vocab)}

# Save vocab
with open("data/mini_data/vocab.pkl", "wb") as f:
    pickle.dump(word2idx, f)

print(f"Vocab size: {len(word2idx)}")


Vocab size: 6147


### Dataset

In [10]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import json
from pathlib import Path

class RecipeDatasetForEval(Dataset):
    def __init__(self, json_path, image_dir, word2idx, transform, max_len=20, mode='dual'):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.image_dir = Path(image_dir)
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len
        self.mode = mode

    def tokenize(self, entries):
        ids = []
        for e in entries:
            for tok in e.lower().split():
                ids.append(self.word2idx.get(tok, self.word2idx['<unk>']))
        ids = ids[:self.max_len] + [self.word2idx['<pad>']] * (self.max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image_path = self.image_dir / item['image']
        image = self.transform(Image.open(image_path).convert('RGB'))

        if self.mode == 'dual':
            tokens = self.tokenize(item['ingredients'])
            return image, tokens, idx

        elif self.mode == 'joint':
            combined = item['ingredients'] + item['instructions']
            tokens = self.tokenize(combined)
            return image, tokens

        else:
            raise ValueError("Mode must be either 'dual' or 'joint'")

### Recall Class module

In [13]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import pairwise_distances
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

@torch.no_grad()
def compute_recall_at_k_dual(model, dataloader, k_list=[1, 5, 10]):
    model.eval()
    all_img_embs, all_txt_embs = [], []

    for img, txt, _ in tqdm(dataloader, desc="Dual Model Eval"):
        img, txt = img.cuda(), txt.cuda()
        img_emb, txt_emb = model(img, txt)
        all_img_embs.append(img_emb)
        all_txt_embs.append(txt_emb)

    img_matrix = torch.cat(all_img_embs)
    txt_matrix = torch.cat(all_txt_embs)
    sims = img_matrix @ txt_matrix.T
    ranks = sims.argsort(dim=1, descending=True)
    correct = torch.arange(len(ranks)).unsqueeze(1).to(ranks.device)

    return {
    "recalls": {f"Recall@{k}": (ranks[:, :k] == correct).any(dim=1).float().mean().item() for k in k_list},
    "img_embs": img_matrix,
    "txt_embs": txt_matrix
}

@torch.no_grad()
def compute_recall_at_k_joint(image_encoder, text_encoder, dataloader, k_list=[1, 5, 10]):
    image_encoder.eval()
    text_encoder.eval()
    all_img_embs, all_txt_embs = [], []

    for img, txt in tqdm(dataloader, desc="Joint Encoder Eval"):
        img, txt = img.cuda(), txt.cuda()
        img_emb = image_encoder(img)
        txt_emb = text_encoder(img_emb, txt)
        all_img_embs.append(img_emb)
        all_txt_embs.append(txt_emb)

    img_matrix = torch.cat(all_img_embs)
    txt_matrix = torch.cat(all_txt_embs)
    sims = img_matrix @ txt_matrix.T
    ranks = sims.argsort(dim=1, descending=True)
    correct = torch.arange(len(ranks)).unsqueeze(1).to(ranks.device)

    return {
    "recalls": {f"Recall@{k}": (ranks[:, :k] == correct).any(dim=1).float().mean().item() for k in k_list},
    "img_embs": img_matrix,
    "txt_embs": txt_matrix
}

@torch.no_grad()
def visualize_embeddings(img_embs, txt_embs):
    all_embs = torch.cat([img_embs, txt_embs], dim=0).cpu().numpy()
    labels = ["Image"] * img_embs.size(0) + ["Text"] * txt_embs.size(0)
    tsne = TSNE(n_components=2, perplexity=30)
    reduced = tsne.fit_transform(all_embs)
    
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, alpha=0.7)
    plt.title("t-SNE of Joint Embeddings")
    plt.show()

### Train with Hard Negatives + Triplet Loss

In [14]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import json, pickle, random
from PIL import Image
from pathlib import Path
import os

from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, TransformerEncoder, BertIngredientEncoder

import numpy as np
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Config
BATCH_SIZE = 64
EPOCHS = 5
EMBED_DIM = 256
LR = 5e-5
WGT_DECAY = 3e-5
DROPOUT_RATE = 0.5
HARD_NGTV_EPS = 0.05
TEMPERATURE = 0.05
VISION_ENCODER_NAME = 'resnet50'        # 'resnet18', 'resnet50', 'vit_b_16', 'efficientnet_b0'
VISION_TUNE_LAST_N_BLOCK = 0           # set to 0 unless wish to fine-tune pre-trained vision encoder
CHECKPOINT_PATH = "checkpoints"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# Dataset
class HardNegativeDataset(torch.utils.data.Dataset):
    def __init__(self, json_path, image_dir, word2idx, transform, max_len=20, mode="joint",hard_negatives=None):
        with open(json_path, "r") as f:
            self.data = json.load(f)
        self.image_dir = Path(image_dir)
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len
        self.mode = mode
        self.hard_negatives = hard_negatives

    def tokenize(self, texts):
        ids = []
        for t in texts:
            for token in t.lower().split():
                ids.append(self.word2idx.get(token, self.word2idx["<unk>"]))
        ids = ids[:self.max_len] + [self.word2idx['<pad>']] * (self.max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        anchor = self.data[idx]
        image = self.transform(Image.open(self.image_dir / anchor["image"]).convert("RGB"))
        pos = self.tokenize(anchor["ingredients"] + anchor["instructions"]) if self.mode == "joint" else self.tokenize(anchor["ingredients"])

        # HARD NEGATIVE MINING (if available)
        if self.hard_negatives is not None:
            neg_idx = self.hard_negatives[idx]
        else:
            # fallback to random or semi-hard
            anchor_ingr = anchor["ingredients"][0].lower().split()[0]
            neg_idx = next((i for i in range(len(self.data)) if i != idx and anchor_ingr in self.data[i]["ingredients"][0].lower()), idx)

        neg = self.tokenize(self.data[neg_idx]["ingredients"] + self.data[neg_idx]["instructions"]) if self.mode == "joint" else self.tokenize(self.data[neg_idx]["ingredients"])
        return image, pos, neg

# Triplet Loss
def triplet_loss(anchor, positive, negative, margin=0.2):
    # return torch.clamp(margin + F.cosine_similarity(anchor, negative) - F.cosine_similarity(anchor, positive), min=0).mean()      # hinge version - clear separation but has hard threshold at 0
    return F.softplus(F.cosine_similarity(anchor, negative) - F.cosine_similarity(anchor, positive) + margin).mean()                # softer version - loss is always +ve and non-zero gradient (better for noisy or smaller dataset)

def info_nce_loss(image_embs, text_embs, temperature=0.07):
    """
    Compute symmetric InfoNCE loss for joint image-text embeddings.

    Args:
        image_embs: [B, D] image embeddings
        text_embs:  [B, D] text embeddings
        temperature: scaling factor (typically 0.05 to 0.1)

    Returns:
        Symmetric contrastive loss
    """
    # Normalize to unit vectors (important for cosine sim)
    image_embs = F.normalize(image_embs, dim=1)
    text_embs = F.normalize(text_embs, dim=1)

    # Cosine similarity matrix [B, B]
    logits_per_image = image_embs @ text_embs.T / temperature
    logits_per_text = text_embs @ image_embs.T / temperature

    labels = torch.arange(image_embs.size(0), device=image_embs.device)

    # Cross entropy loss
    loss_i2t = F.cross_entropy(logits_per_image, labels)
    loss_t2i = F.cross_entropy(logits_per_text, labels)

    return (loss_i2t + loss_t2i) / 2

def compute_sim_matrix_batched(embs, batch_size=512):
    embs = embs.cpu()
    N = embs.size(0)
    sim_matrix = torch.empty((N, N))
    for i in tqdm(range(0, N, batch_size), desc="Chunked sim_matrix"):
        i_end = min(i + batch_size, N)
        for j in range(0, N, batch_size):
            j_end = min(j + batch_size, N)
            sim = F.cosine_similarity(
                embs[i:i_end].unsqueeze(1),
                embs[j:j_end].unsqueeze(0),
                dim=-1
            )
            sim_matrix[i:i_end, j:j_end] = sim
    return sim_matrix


def mine_hard_negatives(image_encoder, text_encoder, dataset, batch_size=64, device="cuda", k=10, mode="joint", epsilon=0.05):
    image_encoder.eval()
    text_encoder.eval()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    img_embs, txt_embs = [], []

    with torch.no_grad():
        for img, pos, _ in tqdm(dataloader, desc="Mining Embeddings"):
            img, pos = img.to(device), pos.to(device)
            img_emb = image_encoder(img)
            if mode == "joint":
                txt_emb = text_encoder(img_emb, pos)
            else:
                txt_emb = text_encoder(pos)
            img_embs.append(img_emb.cpu())
            txt_embs.append(txt_emb.cpu())

    img_embs = torch.cat(img_embs)  # [N, D]
    txt_embs = torch.cat(txt_embs)  # [N, D]

    N = len(txt_embs)
    CHUNK_SIZE = 256  # lower if needed

    if mode == "dual":
        sim_matrix = []
        for i in tqdm(range(0, N, CHUNK_SIZE), desc="Computing sim_matrix"):
            chunk = img_embs[i:i+CHUNK_SIZE].to(device)     # image queries
            ref = txt_embs.to(device)                       # all text candidates
            sims = F.cosine_similarity(chunk.unsqueeze(1), ref.unsqueeze(0), dim=-1)  # [C, N]
            sim_matrix.append(sims.cpu())
        sim_matrix = torch.cat(sim_matrix, dim=0)           # [N, N]
    else:
        sim_matrix = []
        txt_embs_cpu = txt_embs.cpu()  # ensure full embeddings are on CPU
        for i in tqdm(range(0, N, CHUNK_SIZE), desc="Computing sim_matrix"):
            chunk = txt_embs_cpu[i:i+CHUNK_SIZE]  # already on CPU
            ref = txt_embs_cpu  # already on CPU
            sims = F.cosine_similarity(chunk.unsqueeze(1), ref.unsqueeze(0), dim=-1)  # [C, N]
            sim_matrix.append(sims)

        sim_matrix = torch.cat(sim_matrix, dim=0)  # [N, N]
    sim_matrix.fill_diagonal_(-1e9)

    # Semi-hard selection: Not closest, not too easy
    hard_negatives = []
    for i in range(N):
        if mode == "dual":
            pos_sim = F.cosine_similarity(img_embs[i].unsqueeze(0), txt_embs[i].unsqueeze(0), dim=-1).item()
        else:
            pos_sim = F.cosine_similarity(txt_embs[i].unsqueeze(0), img_embs[i].unsqueeze(0), dim=-1).item()
        candidates = sim_matrix[i]
        semi_hard_mask = (candidates < pos_sim) & (candidates > pos_sim - epsilon)
        valid = torch.nonzero(semi_hard_mask).squeeze(-1)

        if len(valid) == 0:
            # fallback to random if no semi-hard found
            neg_idx = random.choice([j for j in range(N) if j != i])
        else:
            neg_idx = valid[torch.randint(0, len(valid), (1,))].item()
        hard_negatives.append(neg_idx)

    return hard_negatives

# Train Dual or Joint
def train(mode="joint", dropout=0.1, eval_loader=None, temperature=0.07):
    with open("data/mini_data/vocab.pkl", "rb") as f:
        word2idx = pickle.load(f)

    # train_transform = transforms.Compose([
    #     transforms.Resize((224, 224)),
    #     transforms.ToTensor(),
    # ])

    best_recall1 = 0.0
    best_model_state = None

    # Data augmentation on training set only to combat overfitting (can switch to easy one like above)
    train_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
    ])

    dataset = HardNegativeDataset("data/mini_data/train_recipes_stratified.json", "data/mini_data", word2idx, train_transform, mode=mode)
    dataloader = DataLoader(
                    dataset, 
                    batch_size=BATCH_SIZE, 
                    shuffle=True, 
                    drop_last=True,
                    num_workers=0,          # adjust based on CPU cores
                    pin_memory=True        # speeds up host-to-GPU transfers
    )

    # Check to see if eval_loader is present
    if eval_loader is None:
        raise ValueError("eval_loader must be provided")

    image_encoder = ImageEncoder(
                                embed_dim=EMBED_DIM,
                                model_name=VISION_ENCODER_NAME,
                                tune_last_n_blocks=VISION_TUNE_LAST_N_BLOCK,
                                dropout=dropout
                            ).cuda()
    # Print trainable parameters for vision encoders
    image_encoder.print_trainable_layers()

    if mode == "joint":
        text_encoder = JointEncoderWithCrossAttention(vocab_size=len(word2idx), embed_dim=EMBED_DIM, dropout=dropout).cuda()
    else:
        text_encoder = TransformerEncoder(vocab_size=len(word2idx), embed_dim=EMBED_DIM, dropout=dropout).cuda()

    optimizer = torch.optim.AdamW(list(image_encoder.parameters()) + list(text_encoder.parameters()), lr=LR, weight_decay=WGT_DECAY)

    all_losses = []
    all_recalls = []

    for epoch in range(EPOCHS):
        if epoch > 0 and epoch % 1 == 0:       # adjust frequency of hard negative mining
            print("Re-mining hard negatives...")
            # mine top 10 hard negatives then random sample as opposed to just top 1 for stability
            dataset.hard_negatives = mine_hard_negatives(image_encoder, text_encoder, dataset, k=10, mode=mode, epsilon=HARD_NGTV_EPS)       
            # pass
        
        image_encoder.train()
        text_encoder.train()
        total_loss = 0

        for batch_idx, (img, pos, neg) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}")):
            img, pos, neg = img.cuda(), pos.cuda(), neg.cuda()
            img_emb = image_encoder(img)
            if mode == "joint":
                pos_emb = text_encoder(img_emb, pos)
                neg_emb = text_encoder(img_emb, neg)
            else:
                pos_emb = text_encoder(pos)
                neg_emb = text_encoder(neg)
            # Debug print - can be removed later (to see how wide should be set on triplet loss margin)
            if batch_idx % 50 == 0:  # Only print every 50 steps
                with torch.no_grad():
                    cos_pos = F.cosine_similarity(img_emb, pos_emb)
                    cos_neg = F.cosine_similarity(img_emb, neg_emb)
                    print(f"[Epoch {epoch+1} | Step {batch_idx}] avg_cos_pos={cos_pos.mean():.4f} | avg_cos_neg={cos_neg.mean():.4f}")

            loss = info_nce_loss(img_emb, pos_emb, temperature=temperature)  # assuming pos_emb is the paired text embeddings
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        all_losses.append(avg_loss)
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

        # Evaluate Recall@K
        if mode == "joint":
            recall = compute_recall_at_k_joint(image_encoder, text_encoder, eval_loader)
        else:
            class DualWrapper(torch.nn.Module):
                def __init__(self, img_enc, txt_enc):
                    super().__init__()
                    self.img_enc = img_enc
                    self.txt_enc = txt_enc
                def forward(self, img, txt):
                    return self.img_enc(img), self.txt_enc(txt)
            recall = compute_recall_at_k_dual(DualWrapper(image_encoder, text_encoder), eval_loader)
        
        all_recalls.append(recall)
        print(f"Epoch {epoch+1} Recall@K: {recall}")

        # only print chart of TSNE embedding visualization (can be commented out)
        if (epoch + 1) % 5 == 0:
            if isinstance(recall, dict) and "img_embs" in recall:
                visualize_embeddings(recall["img_embs"], recall["txt_embs"])

        # Save current model every epoch
        epoch_ckpt_path = Path(CHECKPOINT_PATH) / f"{mode}_encoder_epoch{epoch+1}.pth"
        torch.save({
            'image_encoder': image_encoder.state_dict(),
            'text_encoder': text_encoder.state_dict(),
            'config': {
                'mode': mode,
                'model_name': VISION_ENCODER_NAME,
                'tune_last_n_blocks': VISION_TUNE_LAST_N_BLOCK,
                'embed_dim': EMBED_DIM,
                'temperature': temperature
            }
        }, epoch_ckpt_path)

        # Save best model based on Recall@1
        curr_recall1 = recall["recalls"]["Recall@1"]
        if curr_recall1 > best_recall1:
            best_recall1 = curr_recall1
            best_model_state = {
                'epoch': epoch + 1,
                'image_encoder': image_encoder.state_dict(),
                'text_encoder': text_encoder.state_dict(),
                'optimizer': optimizer.state_dict(),
                'config': {
                    'mode': mode,
                    'model_name': VISION_ENCODER_NAME,
                    'tune_last_n_blocks': VISION_TUNE_LAST_N_BLOCK,
                    'embed_dim': EMBED_DIM,
                    'temperature': temperature
                },
                'best_recall1': best_recall1
            }
            torch.save(best_model_state, Path(CHECKPOINT_PATH) / f"{mode}_encoder_best.pth")
            print(f"[Epoch {epoch+1}] best model saved - Recall@1 = {best_recall1:.4f}")

    # Plot Loss
    plt.figure()
    plt.plot(range(1, EPOCHS+1), all_losses, label="Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Over Epochs")
    plt.legend()
    plt.savefig(f"{mode}_loss_plot.png")

    # Plot Recall@K
    for k in [1, 5, 10]:
        plt.plot([r["recalls"][f"Recall@{k}"] for r in all_recalls], label=f"Recall@{k}")
    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.title("Recall@K Over Epochs")
    plt.legend()
    plt.savefig(f"{mode}_recall_plot.png")

if __name__ == "__main__":
    with open("data/mini_data/vocab.pkl", "rb") as f:
        word2idx = pickle.load(f)

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    eval_dataset = RecipeDatasetForEval("data/mini_data_val/val_recipes_stratified.json", "data/mini_data_val", word2idx, val_transform, mode="joint")
    eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)
    train(mode="joint", dropout=DROPOUT_RATE, eval_loader=eval_loader, temperature=TEMPERATURE)        # or train(mode="dual")


C:\Users\Me\Desktop\New folder (2)\7643_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Me/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:09<00:00, 11.4MB/s]



Trainable layers in resnet50:
  fc.0.weight
  fc.0.bias


Epoch 1:   0%|          | 0/156 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'data\\mini_data\\8ca2985d5a.jpg'

### Top K Retrieval and storage

In [ ]:
import torch
import json
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, TransformerEncoder
import pickle
from pathlib import Path

@torch.no_grad()
def run_retrieval_and_save_results(
    model_mode="joint",
    checkpoint_path="checkpoints/joint_encoder_best.pth",
    val_json="data/mini_data_val/val_recipes_stratified.json",
    val_img_dir="data/mini_data_val",
    vocab_path="data/mini_data/vocab.pkl",
    outfile="retrieval_results.json",
    topk=5,
    save_failures_only=False,
    failure_output="retrieval_failures.json"
):
    # Load model
    image_encoder = ImageEncoder(embed_dim=512).cuda()

    checkpoint = torch.load(checkpoint_path)
    vocab_size = checkpoint['text_encoder']['text_embed.weight'].size(0)

    if model_mode == "joint":
        text_encoder = JointEncoderWithCrossAttention(vocab_size=vocab_size, embed_dim=512).cuda()
    else:
        text_encoder = TransformerEncoder(vocab_size=vocab_size, embed_dim=512).cuda()

    image_encoder.load_state_dict(checkpoint['image_encoder'])
    text_encoder.load_state_dict(checkpoint['text_encoder'])
    image_encoder.eval()
    text_encoder.eval()

    # Load data
    with open(vocab_path, "rb") as f:
        word2idx = pickle.load(f)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    eval_dataset = RecipeDatasetForEval(
        json_path=val_json,
        image_dir=val_img_dir,
        word2idx=word2idx,
        transform=transform,
        mode=model_mode
    )

    eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

    image_embeddings = []
    text_embeddings = []
    recipe_entries = []
    image_ids = []
    gt_indices = []

    for batch_idx, (images, tokens) in enumerate(tqdm(eval_loader, desc="Encoding")):
        images, tokens = images.cuda(), tokens.cuda()
        img_emb = image_encoder(images)
        if model_mode == "joint":
            txt_emb = text_encoder(img_emb, tokens)
        else:
            txt_emb = text_encoder(tokens)

        image_embeddings.append(img_emb)
        text_embeddings.append(txt_emb)

        for i in range(images.size(0)):
            entry = eval_dataset.data[batch_idx * eval_loader.batch_size + i]
            recipe_entries.append(entry)
            image_ids.append(entry["image"])
            gt_indices.append(batch_idx * eval_loader.batch_size + i)  # assuming aligned order

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    sims = image_embeddings @ text_embeddings.T
    values, indices = torch.topk(sims, k=topk, dim=1)

    results = []
    failures = []

    for i, (img_id, top_idx, gt_idx) in enumerate(zip(image_ids, indices, gt_indices)):
        top_recipes = [recipe_entries[j] for j in top_idx.tolist()]
        hit = gt_idx in top_idx.tolist()
        entry = {
            "query_image": img_id,
            "top_recipes": top_recipes,
            "match_found": hit,
            "ground_truth_idx": gt_idx,
            "retrieved_indices": top_idx.tolist()
        }
        results.append(entry)
        if not hit:
            failures.append(entry)

    with open(outfile, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved retrieval results to {outfile}")

    if save_failures_only:
        with open(failure_output, "w") as f:
            json.dump(failures, f, indent=2)
        print(f"Saved {len(failures)} failures to {failure_output}")


In [ ]:
from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, TransformerEncoder
from torch.utils.data import DataLoader
from torchvision import transforms
import torch, pickle, json


# Config
MODE = "joint"
VAL_JSON = "data/mini_data_val/val_recipes_stratified.json"
VAL_IMG_DIR = "data/mini_data_val"
VOCAB_PATH = "data/mini_data/vocab.pkl"
CHECKPOINT_PATH = f"checkpoints/{MODE}_encoder_best.pth"
TOPK = 5
OUTFILE = "retrieval_results.json"

# Load Vocab & Transform
with open(VOCAB_PATH, "rb") as f:
    word2idx = pickle.load(f)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load Dataset
eval_dataset = RecipeDatasetForEval(
    json_path=VAL_JSON,
    image_dir=VAL_IMG_DIR,
    word2idx=word2idx,
    transform=transform,
    mode=MODE
)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# Load Model
checkpoint = torch.load(CHECKPOINT_PATH)
vocab_size = checkpoint['text_encoder']['text_embed.weight'].size(0)

image_encoder = ImageEncoder(embed_dim=512).cuda()
if MODE == "joint":
    text_encoder = JointEncoderWithCrossAttention(vocab_size=vocab_size, embed_dim=512).cuda()
else:
    text_encoder = TransformerEncoder(vocab_size=vocab_size, embed_dim=512).cuda()

image_encoder.load_state_dict(checkpoint['image_encoder'])
text_encoder.load_state_dict(checkpoint['text_encoder'])
image_encoder.eval()
text_encoder.eval()

# Run Retrieval + Save Results
run_retrieval_and_save_results(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    dataloader=eval_loader,
    dataset=eval_dataset,
    mode=MODE,
    topk=TOPK,
    output_path=OUTFILE,
    only_failures=False,     # optional - only save failed queries

)

In [ ]:
# # Optional - show failed retrieval


# show_retrieval_failures(
#     retrieval_path="retrieval_results.json",
#     dataset_dir="data/mini_data_val", 
#     num_failures=5
# )

In [ ]:
# GEMMA prompt for testing/demos/custom prompt runs

import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load Gemma Model
model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
model.eval()

# Generate from Prompt
def generate_instructions(ingredients, title=None, max_length=256):
    if title:
        prompt = f"Recipe Title: {title}\nIngredients: {', '.join(ingredients)}\nInstructions:"
    else:
        prompt = f"Ingredients: {', '.join(ingredients)}\nInstructions:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Batch Generate from Top-K Retrievals
def generate_from_topk(topk_json_path, output_path="generated_topk.json"):
    with open(topk_json_path, "r") as f:
        topk_data = json.load(f)

    generations = []
    for entry in topk_data:
        query_img = entry["query_image"]
        for i, candidate in enumerate(entry["top_recipes"]):
            ingredients = candidate["ingredients"]
            title = candidate.get("title", "")
            gen = generate_instructions(ingredients, title)
            generations.append({
                "query_image": query_img,
                "rank": i + 1,
                "title": title,
                "ingredients": ingredients,
                "generated_instructions": gen
            })

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(generations, f, indent=2)
    print(f"Saved {len(generations)} generations to {output_path}")

# Run
if __name__ == "__main__":
    generate_from_topk("retrieval_results.json")



### Generation Evaluation and Re-Ranking

In [ ]:
# After gemma_gen_topk is being ran

import json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import matplotlib.pyplot as plt

# Config
GENERATED_PATH = "generated_topk.json"
GROUNDTRUTH_PATH = "data/mini_data_val/val_recipes_stratified.json"
OUT_PATH = "reranked_generations.json"
BLEU_WEIGHT = (0.25, 0.25, 0.25, 0.25)

# Load
with open(GENERATED_PATH, "r") as f:
    generated = json.load(f)
with open(GROUNDTRUTH_PATH, "r") as f:
    groundtruth = {r["image"]: r["instructions"] for r in json.load(f)}

# Model for cosine similarity
model = SentenceTransformer("all-MiniLM-L6-v2")

results = []
bleu_scorer = SmoothingFunction().method1

bleu_scores = []
cosine_scores = []

for item in tqdm(generated, desc="Evaluating and Reranking"):
    img_id = item["query_image"]
    gt_instr = " ".join(groundtruth.get(img_id, []))
    gt_emb = model.encode(gt_instr, convert_to_tensor=True)

    scored = []
    for gen in item["topk_generations"]:
        gen_text = gen["generated_instructions"]
        bleu = sentence_bleu([gt_instr.split()], gen_text.split(), weights=BLEU_WEIGHT, smoothing_function=bleu_scorer)
        gen_emb = model.encode(gen_text, convert_to_tensor=True)
        cosine = util.cos_sim(gt_emb, gen_emb).item()

        score = 0.5 * bleu + 0.5 * cosine
        gen["bleu"] = bleu
        gen["cosine"] = cosine
        gen["combined_score"] = score
        scored.append(gen)

    scored.sort(key=lambda x: x["combined_score"], reverse=True)
    results.append({
        "query_image": img_id,
        "topk_generations": scored,
        "best_generation": scored[0] if scored else None
    })

    if scored:
        bleu_scores.append(scored[0]["bleu"])
        cosine_scores.append(scored[0]["cosine"])

# Save results
with open(OUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Reranked generations with BLEU + Cosine to {OUT_PATH}")

# Summary Stats
avg_bleu = sum(bleu_scores) / len(bleu_scores)
avg_cosine = sum(cosine_scores) / len(cosine_scores)
print(f"\n📉 Average BLEU: {avg_bleu:.4f}")
print(f"📉 Average Cosine Similarity: {avg_cosine:.4f}")

# Visualization
plt.figure(figsize=(8, 5))
plt.plot(bleu_scores, label="BLEU Scores")
plt.plot(cosine_scores, label="Cosine Similarity")
plt.title("Best Generation Scores per Query")
plt.xlabel("Query Index")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("score_plot.png")
plt.show()


# Print Top-5 Samples by BLEU and Cosine
print("\n Top 5 by BLEU Score:")
top_bleu = sorted(results, key=lambda x: x['best_generation']['bleu'], reverse=True)[:5]
for i, item in enumerate(top_bleu):
    b = item['best_generation']
    print(f"{i+1}. {item['query_image']} | BLEU: {b['bleu']:.4f} | Cosine: {b['cosine']:.4f}")
    print(f"   ↪ {b['generated_instructions'][:100]}...")

print("\n Top 5 by Cosine Similarity:")
top_cosine = sorted(results, key=lambda x: x['best_generation']['cosine'], reverse=True)[:5]
for i, item in enumerate(top_cosine):
    b = item['best_generation']
    print(f"{i+1}. {item['query_image']} | Cosine: {b['cosine']:.4f} | BLEU: {b['bleu']:.4f}")
    print(f"   ↪ {b['generated_instructions'][:100]}...")